# 모듈 4: 경사하강법(Gradient Descent) 실습 및 증명

이 노트북에서는:
1. 미분(기울기)의 효과를 직접 코드로 체험하고,
2. 수작업 경사하강법과 파이토치 옵티마이저의 결과가 동일함을 증명하며,
3. 학습률(Learning Rate) 변화에 따른 수렴/발산을 시각적으로 확인합니다.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 0. 미분 기초: 두 점 기울기에서 접선 기울기로

함수 $f(x)=x^2$에서 한 점 $x_0=3$의 접선 기울기를 구해봅니다.
두 점 기울기 $(f(x_0+h)-f(x_0))/h$에서 $h$를 0에 가깝게 보내면 미분값에 수렴합니다.

In [ ]:
x0 = 3.0
true_derivative = 2 * x0  # f(x)=x^2 이므로 f'(x)=2x
h_values = [1.0, 0.5, 0.1, 0.01, 0.001]

print("=== 두 점 기울기 -> 접선 기울기(미분) 수렴 확인 ===")
for h in h_values:
    secant_slope = ((x0 + h) ** 2 - x0 ** 2) / h
    print(f"h={h:>6}: 두 점 기울기={secant_slope:>8.6f}")

print(f"이론적 접선 기울기 f'({x0}) = {true_derivative:.6f}")

## 0-1. 과일가게 예제로 체인룰 코드 증명

식: $z=(\text{사과가격}\times\text{개수})\times\text{세율}$

체인룰로
$$\frac{\partial z}{\partial \text{사과가격}}=\text{개수}\times\text{세율}$$
가 나와야 합니다.

In [ ]:
apple_price = torch.tensor(100.0, requires_grad=True)
apple_count = 2.0
tax = 1.1

subtotal = apple_price * apple_count
total_price = subtotal * tax

total_price.backward()  # d(total_price)/d(apple_price)

manual_grad = apple_count * tax
autograd_grad = apple_price.grad.item()

print("=== 과일가게 체인룰 검증 ===")
print(f"최종 금액 z = {total_price.item():.1f}")
print(f"수식(체인룰) 미분값 = {manual_grad:.4f}")
print(f"PyTorch autograd 미분값 = {autograd_grad:.4f}")
print("해석: 사과 1개 가격이 1원 오르면 최종 금액은 2.2원 오른다.")

## 1. 미분의 효과 직접 체험

간단한 함수 $f(x) = x^2$ 의 최적점(최소값)은 $x=0$ 입니다.
시작점 $x=4.0$ 에서 출발하여, 기울기(미분)만으로 최적점을 찾아가는 과정을 봅시다.

$f(x) = x^2$ 의 미분: $\frac{df}{dx} = 2x$
- $x=4$ → 기울기 $= 8$ (양수 = 오르막 = 왼쪽으로 가야 함!)
- $x=-3$ → 기울기 $= -6$ (음수 = 내리막 = 오른쪽으로 가야 함!)
- $x=0$ → 기울기 $= 0$ (평지 = 최적점 도착!)

수치 예시(한 번 더):
- Step 1: $x_1 = 4.0 - 0.1 \times (2 \times 4.0)=3.2$
- Step 2: $x_2 = 3.2 - 0.1 \times (2 \times 3.2)=2.56$

In [ ]:
# =============================================
# A. 수작업 경사하강법 (For 루프로 직접 구현)
# =============================================
x_manual = 4.0  # 시작점
lr = 0.1        # 학습률 (한 걸음 보폭)
manual_history = [x_manual]

print("=== 수작업 경사하강법 (f(x) = x², df/dx = 2x) ===")
for step in range(20):
    gradient = 2 * x_manual                            # 미분값 계산
    x_manual = x_manual - lr * gradient                # 업데이트 규칙: x_new = x_old - η·기울기
    manual_history.append(x_manual)
    if step < 8 or step >= 18:
        print(f"  Step {step+1:2d}: x = {x_manual:8.4f}, 기울기 = {gradient:8.4f}, f(x) = {x_manual**2:8.4f}")
    elif step == 8:
        print(f"  ... (중간 생략) ...")

print(f"\n최종 도달 위치: x = {x_manual:.6f} (목표: 0.0)")
print("→ 미분이 알려주는 기울기 방향으로 반복 이동하여 최적점에 수렴했습니다!")

In [ ]:
# =============================================
# B. 파이토치 옵티마이저 (SGD)로 동일 과정 실행
# =============================================
x_pt = torch.tensor([4.0], requires_grad=True)  # requires_grad=True → 파이토치에게 "이 변수의 미분을 추적해줘" 요청
optimizer = torch.optim.SGD([x_pt], lr=0.1)
pt_history = [x_pt.item()]

print("\n=== PyTorch SGD 옵티마이저 ===")
for step in range(20):
    loss = x_pt ** 2                 # f(x) = x²
    optimizer.zero_grad()            # 이전 기울기 초기화
    loss.backward()                  # 자동 미분 실행 (df/dx = 2x 를 자동 계산)
    optimizer.step()                 # x = x - lr * gradient 자동 업데이트
    pt_history.append(x_pt.item())
    if step < 3:
        print(f"  Step {step+1}: x = {x_pt.item():8.4f}, 자동 계산된 기울기 = {x_pt.grad.item():8.4f}")

print(f"  ...")
print(f"  최종: x = {x_pt.item():.6f}")

In [ ]:
# =============================================
# C. 두 방법의 매 스텝 결과 비교 (완전 일치 증명)
# =============================================
print("\n=== 수작업 vs PyTorch SGD 매 스텝 비교 ===")
print(f"{'Step':>4} | {'수작업 x':>12} | {'PyTorch x':>12} | {'일치':>4}")
print("-" * 48)
for i in range(min(len(manual_history), len(pt_history))):
    match = "✅" if abs(manual_history[i] - pt_history[i]) < 1e-6 else "❌"
    print(f"{i:>4} | {manual_history[i]:>12.6f} | {pt_history[i]:>12.6f} | {match:>4}")

print("\n→ 결론: 파이토치의 SGD 옵티마이저는 우리가 수식으로 배운")
print("  'x_new = x_old - lr × 기울기' 규칙을 그대로 실행하는 것에 불과합니다!")

## 2. 학습률(Learning Rate)의 극적 효과 시각화

학습률을 바꾸면 수렴 속도와 안정성이 어떻게 변하는지 확인합니다.

In [ ]:
learning_rates = [0.01, 0.1, 0.9]
colors = ['blue', 'green', 'red']
labels = ['lr=0.01 (너무 느림)', 'lr=0.1 (적절)', 'lr=0.9 (불안정)']

plt.figure(figsize=(12, 5))

for lr_val, color, label in zip(learning_rates, colors, labels):
    x_test = 4.0
    history = [x_test]
    for _ in range(30):
        grad = 2 * x_test
        x_test = x_test - lr_val * grad
        history.append(x_test)
    plt.plot(history, marker='o', markersize=3, color=color, label=label)

plt.axhline(y=0, color='black', linestyle='--', alpha=0.5, label='최적점 (x=0)')
plt.xlabel('Step', fontsize=12)
plt.ylabel('x 값', fontsize=12)
plt.title('학습률(Learning Rate)에 따른 수렴 패턴 비교', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("→ 파란색(lr=0.01): 안전하지만 매우 느리게 수렴")
print("→ 초록색(lr=0.1):  적절한 속도로 안정적 수렴")
print("→ 빨간색(lr=0.9):  진동하며 불안정, 더 크면 발산(학습 실패)")

## 3. 공식 적용 예제: 1차 함수 최소점 찾기

경사하강법 공식 $	heta \leftarrow 	heta - \eta 
abla J(	heta)$ 를 수치로 적용합니다.

예시 목적함수: $J(	heta)=(	heta-3)^2$ (최소점은 $	heta=3$)


In [ ]:
theta = 10.0
lr = 0.1

for step in range(1, 11):
    grad = 2 * (theta - 3)  # d/dtheta (theta-3)^2
    theta = theta - lr * grad
    loss = (theta - 3) ** 2
    print(f'step {step:02d}: theta={theta:.4f}, loss={loss:.6f}')


## 4. 은닉층 1개 + 출력층 1개: 체인룰로 미분 전개

구조를 다음처럼 둡니다.

- $z_1 = w_1x + b_1$,  $h=\sigma(z_1)$
- $z_2 = w_2h + b_2$,  $\hat{y}=\sigma(z_2)$
- $L=\frac{1}{2}(\hat{y}-y)^2$

아래 코드에서 **수식 기반 수동 미분값**과 **PyTorch autograd 미분값**이 일치하는지 확인합니다.

In [ ]:
def sigmoid(v):
    return 1 / (1 + torch.exp(-v))

# 샘플 1개와 초기 파라미터
x = 1.5
y = 1.0
w1, b1 = 0.5, -0.4
w2, b2 = -1.2, 0.3
lr = 0.1

# 1) 순전파 (수동)
z1 = w1 * x + b1
h = 1 / (1 + torch.exp(torch.tensor(-z1))).item()
z2 = w2 * h + b2
yhat = 1 / (1 + torch.exp(torch.tensor(-z2))).item()
loss = 0.5 * (yhat - y) ** 2

# 2) 체인룰로 수동 미분
dL_dyhat = (yhat - y)
dyhat_dz2 = yhat * (1 - yhat)
dL_dz2 = dL_dyhat * dyhat_dz2

dL_dw2_manual = dL_dz2 * h
dL_db2_manual = dL_dz2

dL_dh = dL_dz2 * w2
dh_dz1 = h * (1 - h)
dL_dz1 = dL_dh * dh_dz1

dL_dw1_manual = dL_dz1 * x
dL_db1_manual = dL_dz1

print("=== 수동 체인룰 미분 ===")
print(f"dL/dw1 = {dL_dw1_manual:.8f}, dL/db1 = {dL_db1_manual:.8f}")
print(f"dL/dw2 = {dL_dw2_manual:.8f}, dL/db2 = {dL_db2_manual:.8f}")

# 3) autograd 미분
x_t = torch.tensor([x], dtype=torch.float32)
y_t = torch.tensor([y], dtype=torch.float32)
w1_t = torch.tensor([w1], dtype=torch.float32, requires_grad=True)
b1_t = torch.tensor([b1], dtype=torch.float32, requires_grad=True)
w2_t = torch.tensor([w2], dtype=torch.float32, requires_grad=True)
b2_t = torch.tensor([b2], dtype=torch.float32, requires_grad=True)

h_t = sigmoid(w1_t * x_t + b1_t)
yhat_t = sigmoid(w2_t * h_t + b2_t)
loss_t = 0.5 * (yhat_t - y_t) ** 2
loss_t.backward()

print("\n=== autograd 미분 ===")
print(f"dL/dw1 = {w1_t.grad.item():.8f}, dL/db1 = {b1_t.grad.item():.8f}")
print(f"dL/dw2 = {w2_t.grad.item():.8f}, dL/db2 = {b2_t.grad.item():.8f}")

print("\n=== 수동 vs autograd 오차 ===")
print(f"|dw1 차이| = {abs(dL_dw1_manual - w1_t.grad.item()):.12f}")
print(f"|db1 차이| = {abs(dL_db1_manual - b1_t.grad.item()):.12f}")
print(f"|dw2 차이| = {abs(dL_dw2_manual - w2_t.grad.item()):.12f}")
print(f"|db2 차이| = {abs(dL_db2_manual - b2_t.grad.item()):.12f}")

# 4) 경사하강법 1스텝 업데이트 후 손실 감소 확인
with torch.no_grad():
    w1_new = w1_t - lr * w1_t.grad
    b1_new = b1_t - lr * b1_t.grad
    w2_new = w2_t - lr * w2_t.grad
    b2_new = b2_t - lr * b2_t.grad

    h_new = sigmoid(w1_new * x_t + b1_new)
    yhat_new = sigmoid(w2_new * h_new + b2_new)
    loss_new = 0.5 * (yhat_new - y_t) ** 2

print("\n=== 경사하강법 1스텝 결과 ===")
print(f"업데이트 전 손실: {loss_t.item():.8f}")
print(f"업데이트 후 손실: {loss_new.item():.8f}")